In [ ]:
# Import necessary packages
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
import random
import copy
import math


# set seed and compute
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# set parameters
HIST_LEN = 50
NUM_SUP = 7
NUM_INNER = 3
TRM_EPOCHS = 50
BATCH_SIZE = 512
TRM_LR = 1e-3
TEMP = 0.07

CAND_SIZE = 100
NUM_NEG = CAND_SIZE - 1
TOPK = [1, 5, 10]


Using device: cuda


In [ ]:

# Load Data

USER_ITEMS_PATH = "path of user_item interaction file"#(.txt for amazon review, .pkl for steam games)

bert_item_embs = torch.load("output file path of sbert embedding .pt file").to(device)

with open(USER_ITEMS_PATH, "rb") as f:
    user_items = pickle.load(f)

num_items = max(max(seq) for seq in user_items.values()) + 1
EMB_DIM = 384

item_embs_raw = torch.randn(num_items, EMB_DIM) * (1.0 / math.sqrt(EMB_DIM))

user_items = {
    u: [i for i in seq if i < num_items]
    for u, seq in user_items.items() if len(seq) >= 2
}

print(f"Items: {num_items}, Embedding dim: {EMB_DIM}")

Items: 10978, Embedding dim: 384


In [ ]:
# =========================================================
# 4. TRM WITH CORRECTION GATE
# =========================================================
class TRM_Sequential(nn.Module):
    def __init__(self, item_embs, dim, num_sup, num_inner):
        super().__init__()

        #self.item_embs = nn.Embedding.from_pretrained(item_embs, freeze=False)
        self.item_embs = nn.Embedding.from_pretrained(
            bert_item_embs,freeze=False


        self.core = nn.Sequential(
            nn.Linear(dim * 3, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.ReLU(),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )

        self.correction_gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )

        self.num_sup = num_sup
        self.num_inner = num_inner
        self.step_scale = 0.7  

    def forward(self, hist_ids, cand_ids, hist_mask):
        h_embs = self.item_embs(hist_ids)

        mask = hist_mask.unsqueeze(-1)
        x = (h_embs * mask).sum(1) / (mask.sum(1) + 1e-8)

        y = x.clone()
        z = torch.zeros_like(x)

        logits_history = []

        for _ in range(self.num_sup):

            # Inner recursion
            for _ in range(self.num_inner):
                z = self.core(torch.cat([x, y, z], dim=-1))

            # Correction gate
            gate = self.correction_gate(torch.cat([x, y], dim=-1))
            z = z * (1 - gate) + x * gate

            # Update
            delta = self.core(torch.cat([x, y, z], dim=-1))
            y = y + self.step_scale * torch.tanh(delta)

            # Score candidates
            c_embs = self.item_embs(cand_ids)
            logits = torch.einsum("bd,bnd->bn", y, c_embs)
            logits_history.append(logits)

        self.y_final = y.detach()
        return torch.stack(logits_history, dim=0)

In [ ]:
# Candidate Sampling

def sample_candidates(hist_set, target):
    negatives = []
    while len(negatives) < NUM_NEG:
        neg = random.randint(0, num_items - 1)
        if neg != target and neg not in hist_set:
            negatives.append(neg)

    candidates = negatives + [target]
    random.shuffle(candidates)
    return candidates, candidates.index(target)


# Dataset creation

def split_sequences(user_items):
    train = []
    test = []

    for seq in user_items.values():
        train.append(seq[:-1])
        test.append((seq[:-1], seq[-1]))

    return train, test

train_seqs, test_seqs = split_sequences(user_items)


class TRMTrainDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = [seq for seq in sequences if len(seq) >= 2]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        t = np.random.randint(1, len(seq))
        hist = seq[:t]
        tgt = seq[t]
        return hist, tgt


class TRMTestDataset(Dataset):
    def __init__(self, test_pairs):
        self.test_pairs = test_pairs

    def __len__(self):
        return len(self.test_pairs)

    def __getitem__(self, idx):
        hist, tgt = self.test_pairs[idx]
        return hist, tgt



def collate_fn(batch):
    B = len(batch)

    h_ids = torch.zeros(B, HIST_LEN, dtype=torch.long)
    h_mask = torch.zeros(B, HIST_LEN, dtype=torch.float32)
    c_ids = torch.zeros(B, CAND_SIZE, dtype=torch.long)
    t_pos = torch.zeros(B, dtype=torch.long)

    for i, (hist, tgt) in enumerate(batch):
        hist = hist[-HIST_LEN:]
        L = len(hist)

        if L > 0:
            h_ids[i, -L:] = torch.tensor(hist, dtype=torch.long)
            h_mask[i, -L:] = 1.0

        candidates, tgt_idx = sample_candidates(set(hist), tgt)
        c_ids[i] = torch.tensor(candidates, dtype=torch.long)
        t_pos[i] = tgt_idx

    return h_ids, c_ids, h_mask, t_pos


train_loader = DataLoader(
    TRMTrainDataset(train_seqs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    TRMTestDataset(test_seqs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
 # EMA

class EMA:
    def __init__(self, model, decay=0.999):
        self.model = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

In [ ]:

#  Evaluation
@torch.no_grad()
def evaluate(model, loader, ks):
    model.eval()
    hits = {k: 0 for k in ks}
    ndcgs = {k: 0 for k in ks}
    precs = {k: 0 for k in ks}
    total = 0

    for h, c, m, t in loader:
        h, c, m, t = h.to(device), c.to(device), m.to(device), t.to(device)
        logits = model(h, c, m)[-1]
        ranks = logits.argsort(dim=1, descending=True)

        for i in range(h.size(0)):
            rank = (ranks[i] == t[i]).nonzero(as_tuple=True)[0].item() + 1
            for k in ks:
                if rank <= k:
                    hits[k] += 1
                    ndcgs[k] += 1 / math.log2(rank + 1)
                    precs[k] += 1 / k
        total += h.size(0)

    return {
        **{f"HR@{k}": hits[k]/total for k in ks},
        **{f"NDCG@{k}": ndcgs[k]/total for k in ks},
        **{f"Prec@{k}": precs[k]/total for k in ks}
    }

In [ ]:
best_metric = 0
best_epoch = 0
best_state = None
monitor_key = "NDCG@10"




# Training loop

model = TRM_Sequential(item_embs_raw.to(device), EMB_DIM, NUM_SUP, NUM_INNER).to(device)
ema = EMA(model)

optimizer = torch.optim.Adam(model.parameters(), lr=TRM_LR)
criterion = nn.CrossEntropyLoss()

print("\nTraining")

for epoch in range(TRM_EPOCHS):
    model.train()

    for h, c, m, t in train_loader:
        h, c, m, t = h.to(device), c.to(device), m.to(device), t.to(device)
        logits = model(h, c, m)

        loss = sum(
            
            criterion(logits[k] / TEMP, t)
            for k in range(NUM_SUP)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ema.update(model)

    metrics = evaluate(ema.model, test_loader, TOPK)

    current_score = metrics[monitor_key]

    if current_score > best_metric:
        best_metric = current_score
        best_epoch = epoch + 1
        best_state = copy.deepcopy(ema.model.state_dict())

    print(f"Epoch {epoch+1} | " +
          " | ".join(f"{k}: {v:.4f}" for k, v in metrics.items()))


print("\nTraining complete.")



In [ ]:
#Evaluation
print(f"\nBest {monitor_key}: {best_metric:.4f} at epoch {best_epoch}")

ema.model.load_state_dict(best_state)

final_metrics = evaluate(ema.model, test_loader, TOPK)

print("\nFinal evaluation on BEST model:")
for k, v in final_metrics.items():
    print(f"{k}: {v:.4f}")
